# ProPos on SAINT Latent Space

Notebook local pour charger les embeddings SAINT deja enregistres, entrainer l'adaptation tabulaire basee sur ProPos-master sur cet espace latent, puis sauvegarder un artefact compatible avec l'application.

In [12]:
from pathlib import Path

# Resolve the checkout from either the project root or research/notebooks.
PROJECT_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "run.py").is_file() and (p / "backend").is_dir()
)

from datetime import datetime, timezone
import importlib.util
import json

import numpy as np
import pandas as pd
import torch

import sys
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
from saint_loader import load_saint_embeddings_df, get_saint_embedding_artifact_metadata

SEED = 42
K = 12
EPOCHS = 80
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
CONFIDENCE_TEMPERATURE = 1.0
SELECTED_POINT_IDS_PATH = None
RUN_NAME = f"propos_saint_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

MODEL_OUTPUT_DIR = PROJECT_ROOT / '.local/cache/research/propos'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROPOS_MASTER_DIR = PROJECT_ROOT / 'vendor/propos'
PROPOS_TABULAR_ADAPTER_PATH = PROPOS_MASTER_DIR / 'tabular_adapter.py'

spec = importlib.util.spec_from_file_location('propos_master_tabular_adapter', PROPOS_TABULAR_ADAPTER_PATH)
propos_master_tabular_adapter = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(propos_master_tabular_adapter)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print('DEVICE =', DEVICE)
print('PROPOS_TABULAR_ADAPTER_PATH =', PROPOS_TABULAR_ADAPTER_PATH.resolve())
print('MODEL_OUTPUT_DIR =', MODEL_OUTPUT_DIR.resolve())

DEVICE = mps
PROPOS_TABULAR_ADAPTER_PATH = /Users/ayatbouhrir/Desktop/STAAGE/test /Projet/model/ProPos-master/tabular_adapter.py
MODEL_OUTPUT_DIR = /Users/ayatbouhrir/Desktop/STAAGE/test /Projet/model/.propos_cache


In [13]:
def train_propos_on_latent(
    x: np.ndarray,
    *,
    k: int,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    device: torch.device,
    seed: int = 42,
):
    return propos_master_tabular_adapter.train_tabular_propos(
        np.asarray(x, dtype=np.float32),
        num_clusters=k,
        epochs=epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        device=str(device),
        seed=seed,
    )


def infer_saved_propos_on_latent(x: np.ndarray, artifact: dict, *, device: torch.device):
    return propos_master_tabular_adapter.infer_tabular_propos(
        np.asarray(x, dtype=np.float32),
        artifact,
        device=str(device),
    )

In [3]:
saint_metadata = get_saint_embedding_artifact_metadata()
latent_df = load_saint_embeddings_df().copy()

if SELECTED_POINT_IDS_PATH is not None:
    selected_ids = pd.read_csv(SELECTED_POINT_IDS_PATH)['point_id'].astype(int).tolist()
    latent_df = latent_df[latent_df['point_id'].isin(selected_ids)].copy()

latent_cols = [c for c in latent_df.columns if c.startswith('z')]
point_ids = latent_df['point_id'].astype(int).to_numpy()
x_latent = latent_df[latent_cols].to_numpy(dtype=np.float32)

print('Rows =', len(latent_df))
print('Latent cols =', latent_cols)
print('SAINT parquet =', saint_metadata.get('parquetPath'))
print('SAINT model =', saint_metadata.get('modelPath'))
latent_df.head()

Rows = 87281
Latent cols = ['z1', 'z2', 'z3', 'z4', 'z5', 'z6', 'z7', 'z8']
SAINT parquet = /Users/ayatbouhrir/Desktop/STAAGE/test /Projet/model/saint_local/outputs/saint_pfas_latent_mean.parquet
SAINT model = /Users/ayatbouhrir/Desktop/STAAGE/test /Projet/model/saint_local/outputs/saint_pfas_latent_mean.pt


,point_id,z1,z2,z3,z4,z5,z6,z7,z8
0,12622,0.393844,0.179614,-0.227233,-0.136581,-0.159580,0.122966,0.022813,0.209607
1,12623,0.415207,0.194504,-0.227471,-0.141379,-0.165086,0.093925,0.006713,0.218045
2,12624,0.416192,0.168477,-0.243618,-0.124526,-0.148661,0.115028,0.000857,0.227966
3,12625,0.437234,0.204726,-0.247844,-0.130684,-0.183315,0.074567,0.014009,0.212644
4,12626,0.413824,0.192869,-0.224747,-0.150122,-0.161459,0.099297,0.003944,0.219359


In [14]:
result = train_propos_on_latent(
    x_latent,
    k=K,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    device=DEVICE,
    seed=SEED,
)

{
    'training_loss': result['trainingLoss'],
    'backend': result['modelConfig']['backend'],
    'feature_dim': result['modelConfig']['featureDim'],
    'hidden_dim': result['modelConfig']['hiddenDim'],
}

{'training_loss': 2.7334799766540527,
 'backend': 'propos_master_tabular_adapter',
 'feature_dim': 16,
 'hidden_dim': 64}

In [15]:
assignments_df = pd.DataFrame(
    {
        'point_id': point_ids,
        'cluster_id': result['labels'],
        'assignment_confidence': result['pointConfidence'],
    }
)

assignments_df['cluster_id'].value_counts().sort_index()

cluster_id
0      8539
1     35686
2      4434
3      4226
4      8715
5      6070
6      2144
7      6902
8      5041
9      1699
10     3081
11      744
Name: count, dtype: int64

In [16]:
artifact_path = MODEL_OUTPUT_DIR / f'{RUN_NAME}.pt'
assignments_path = MODEL_OUTPUT_DIR / f'{RUN_NAME}_assignments.parquet'
summary_path = MODEL_OUTPUT_DIR / f'{RUN_NAME}_summary.json'

artifact = {
    'run_name': RUN_NAME,
    'artifactPath': str(artifact_path.resolve()),
    'savedAt': datetime.now(timezone.utc).isoformat(),
    'algorithm': 'propos',
    'latentReduction': 'saint',
    'saintMetadata': saint_metadata,
    'latentColumns': latent_cols,
    'pointIds': point_ids,
    'modelState': result['modelState'],
    'modelConfig': result['modelConfig'],
    'centroids': result['centroids'],
    'labels': result['labels'],
    'assignmentConfidence': result['pointConfidence'],
    'trainingLoss': result['trainingLoss'],
    'epochs': result['epochs'],
    'featureMetadata': {
        'stage': 'reduction',
        'requestedReduction': 'saint',
        'effectiveReduction': 'saint',
        'embeddingSource': 'precomputed',
        'outputFeatureNames': latent_cols,
        'outputFeatureCount': len(latent_cols),
        'outputShape': [int(x_latent.shape[0]), int(x_latent.shape[1])],
        'saintArtifact': saint_metadata,
    },
    'trainingConfig': {
        'seed': SEED,
        'k': K,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
    },
    'metrics': {
        'training_loss': result['trainingLoss'],
        'backend': result['modelConfig']['backend'],
    },
}

torch.save(artifact, artifact_path)
assignments_df.to_parquet(assignments_path, index=False)
summary_path.write_text(json.dumps({
    'run_name': RUN_NAME,
    'artifact_path': str(artifact_path.resolve()),
    'assignments_path': str(assignments_path.resolve()),
    'k': K,
    'epochs': EPOCHS,
    'backend': result['modelConfig']['backend'],
    'metrics': artifact['metrics'],
}, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved artifact:', artifact_path.resolve())
print('Saved assignments:', assignments_path.resolve())
print('Saved summary:', summary_path.resolve())

Saved artifact: /Users/ayatbouhrir/Desktop/STAAGE/test /Projet/model/.propos_cache/propos_saint_20260420_150423.pt
Saved assignments: /Users/ayatbouhrir/Desktop/STAAGE/test /Projet/model/.propos_cache/propos_saint_20260420_150423_assignments.parquet
Saved summary: /Users/ayatbouhrir/Desktop/STAAGE/test /Projet/model/.propos_cache/propos_saint_20260420_150423_summary.json


In [17]:
reloaded = torch.load(artifact_path, map_location='cpu', weights_only=False)
inference_preview = infer_saved_propos_on_latent(x_latent[:32], reloaded, device=DEVICE)

print(reloaded['run_name'])
print(reloaded['trainingConfig'])
print(reloaded['modelConfig'])
print({'preview_labels': len(inference_preview['labels']), 'preview_centroids': inference_preview['centroids'].shape})

propos_saint_20260420_150423
{'seed': 42, 'k': 12, 'epochs': 80, 'learning_rate': 0.001, 'weight_decay': 1e-05}
{'inputDim': 8, 'hiddenDim': 64, 'featureDim': 16, 'temperature': 0.5, 'momentum': 0.996, 'noiseStd': 0.03, 'dropoutProb': 0.05, 'backend': 'propos_master_tabular_adapter'}
{'preview_labels': 32, 'preview_centroids': (12, 16)}
